In [1]:
from pyspark.ml.feature import HashingTF, IDF, Tokenizer
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession\
        .builder\
        .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/08 19:49:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/04/08 19:49:32 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
sentenceData = spark.createDataFrame([
    (0.0, "Hi I heard about Spark"),
    (0.0, "I wish Java could use case classes"),
    (0.0, "Logistic regression models are neat")
], ["labels", "sentence"])

In [4]:
sentenceData.show()

+------+--------------------+
|labels|            sentence|
+------+--------------------+
|   0.0|Hi I heard about ...|
|   0.0|I wish Java could...|
|   0.0|Logistic regressi...|
+------+--------------------+



In [6]:
tokenizer = Tokenizer(inputCol="sentence", outputCol="words")

In [7]:
wordsData = tokenizer.transform(sentenceData)

In [8]:
wordsData.show()

+------+--------------------+--------------------+
|labels|            sentence|               words|
+------+--------------------+--------------------+
|   0.0|Hi I heard about ...|[hi, i, heard, ab...|
|   0.0|I wish Java could...|[i, wish, java, c...|
|   0.0|Logistic regressi...|[logistic, regres...|
+------+--------------------+--------------------+



In [9]:
hashingTF = HashingTF(inputCol="words", outputCol="rawFeatures", numFeatures=20)

In [10]:
featurizedData = hashingTF.transform(wordsData)

In [11]:
featurizedData['rawFeatures']

Column<'rawFeatures'>

In [ ]:
#alternatively, countVectorizer can also be used  to get term frequency vectors

In [12]:
idf = IDF(inputCol="rawFeatures", outputCol="features")

In [13]:
idfModel = idf.fit(featurizedData)

In [14]:
rescaledData = idfModel.transform(featurizedData)

In [17]:
rescaledData.select("labels", "features").show()

+------+--------------------+
|labels|            features|
+------+--------------------+
|   0.0|(20,[6,8,13,16],[...|
|   0.0|(20,[0,2,7,13,15,...|
|   0.0|(20,[3,4,6,11,19]...|
+------+--------------------+



In [18]:
spark.stop()